In [ ]:
# =============================
# INSTALL DEPENDENCIES
# =============================
!pip install pandas ipywidgets folium

# =============================
# IMPORTS
# =============================
import pandas as pd
from collections import deque
import heapq
import time
import ipywidgets as widgets
from IPython.display import display, HTML
import folium
from folium.plugins import AntPath

# =============================
# GRAPH DATA
# =============================
data = {
    'From': [
        'Karachi', 'Karachi',
        'Hyderabad', 'Hyderabad',
        'Sukkur', 'Sukkur',
        'Multan', 'Multan',
        'Lahore', 'Lahore',
        'Islamabad', 'Peshawar',
        'Quetta'
    ],
    'To': [
        'Hyderabad', 'Sukkur',
        'Sukkur', 'Multan',
        'Multan', 'Quetta',
        'Lahore', 'Islamabad',
        'Islamabad', 'Peshawar',
        'Peshawar', 'Islamabad',
        'Karachi'
    ],
    'Cost': [
        150, 400,
        200, 500,
        300, 600,
        350, 400,
        300, 180,
        250, 200,
        700
    ],
    'Time': [
        2, 5,
        3, 6,
        4, 7,
        5, 6,
        4, 2,
        3, 2,
        8
    ]
}

df = pd.DataFrame(data)

# =============================
# COORDINATES
# =============================
coords = {
    "Karachi": (24.8607, 67.0011),
    "Hyderabad": (25.3960, 68.3578),
    "Sukkur": (27.7052, 68.8574),
    "Multan": (30.1575, 71.5249),
    "Lahore": (31.5204, 74.3587),
    "Islamabad": (33.6844, 73.0479),
    "Peshawar": (34.0151, 71.5249),
    "Quetta": (30.1798, 66.9750)
}

# =============================
# GRAPH BUILD
# =============================
graph = {}
time_graph = {}

for _, row in df.iterrows():
    graph.setdefault(row['From'], []).append((row['To'], row['Cost']))
    graph.setdefault(row['To'], []).append((row['From'], row['Cost']))

    time_graph.setdefault(row['From'], []).append((row['To'], row['Time']))
    time_graph.setdefault(row['To'], []).append((row['From'], row['Time']))

# =============================
# BFS
# =============================
def bfs(start, goal):
    queue = deque([[start]])
    visited = set()

    while queue:
        path = queue.popleft()
        node = path[-1]

        if node == goal:
            return path

        if node not in visited:
            visited.add(node)
            for neighbor, _ in graph.get(node, []):
                queue.append(path + [neighbor])

    return None

# =============================
# DFS
# =============================
def dfs(start, goal, path=None, visited=None):
    if path is None:
        path = []
    if visited is None:
        visited = set()

    path = path + [start]
    visited.add(start)

    if start == goal:
        return path

    for neighbor, _ in graph.get(start, []):
        if neighbor not in visited:
            result = dfs(neighbor, goal, path, visited)
            if result:
                return result

    return None

# =============================
# UCS (BEST PATH)
# =============================
def ucs(start, goal):
    pq = [(0, start, [], 0)]  # cost, node, path, time
    visited = set()

    while pq:
        cost, node, path, ttime = heapq.heappop(pq)
        path = path + [node]

        if node == goal:
            return path, cost, ttime

        if node not in visited:
            visited.add(node)

            for i, (neighbor, weight) in enumerate(graph.get(node, [])):
                travel_time = time_graph[node][i][1]
                heapq.heappush(
                    pq,
                    (cost + weight, neighbor, path, ttime + travel_time)
                )

    return None, float('inf'), float('inf')

# =============================
# DIRECTIONS
# =============================
def directions(path):
    print("\n🧭 DIRECTIONS:")
    for i in range(len(path) - 1):
        print(f"➡️ {path[i]} → {path[i+1]}")
    print("🏁 Destination reached!")

# =============================
# REAL-TIME SIMULATION
# =============================
def simulate_travel(path):
    print("\n🚗 LIVE TRAVEL SIMULATION STARTED...\n")

    total_time = 0

    for i in range(len(path) - 1):
        a = path[i]
        b = path[i + 1]

        for j, (n, t) in enumerate(time_graph[a]):
            if n == b:
                travel_time = t
                break

        total_time += travel_time

        print(f"🚗 Moving: {a} → {b}")
        print(f"⏱️ Travel Time: {travel_time} hrs")
        print(f"📍 Now at: {b}\n")

        time.sleep(1)  # simulation delay

    print("🏁 ARRIVED!")
    print(f"⏱️ Total Travel Time: {total_time} hrs")

# =============================
# MAP WITH ANIMATION
# =============================
def show_map(path, total_time=None):
    m = folium.Map(location=[30.0, 70.0], zoom_start=5)

    # -------------------
    # CITY MARKERS (CLEAN GUI STYLE)
    # -------------------
    for city, (lat, lon) in coords.items():
        folium.Marker(
            [lat, lon],
            popup=f"""
            <b>{city}</b>
            """,
            tooltip=city,
            icon=folium.Icon(color="blue")
        ).add_to(m)

    if not path:
        return m

    route = [coords[c] for c in path if c in coords]

    # -------------------
    # START / END MARKERS
    # -------------------
    folium.Marker(
        route[0],
        popup=f"<b>START:</b> {path[0]}",
        icon=folium.Icon(color="green")
    ).add_to(m)

    folium.Marker(
        route[-1],
        popup=f"<b>END:</b> {path[-1]}",
        icon=folium.Icon(color="red")
    ).add_to(m)

    # -------------------
    # ANIMATED ROUTE
    # -------------------
    AntPath(
        locations=route,
        color="red",
        weight=5,
        delay=600
    ).add_to(m)

    # ❌ REMOVED: TIME LABEL (as you requested)

    return m

    # -------------------
    # START / END MARKERS
    # -------------------
    folium.Marker(
        route[0],
        popup=f"START: {path[0]}",
        icon=folium.Icon(color="green")
    ).add_to(m)

    folium.Marker(
        route[-1],
        popup=f"END: {path[-1]}",
        icon=folium.Icon(color="red")
    ).add_to(m)

    # -------------------
    # ANIMATED ROUTE
    # -------------------
    AntPath(
        locations=route,
        color="red",
        weight=5,
        delay=600
    ).add_to(m)

    # -------------------
    # TIME LABEL ON ROUTE
    # -------------------
    if total_time is not None:
        mid_index = len(route) // 2
        mid_point = route[mid_index]

        folium.Marker(
            mid_point,
            icon=folium.DivIcon(
                html=f"""
                <div style="
                    font-size: 14px;
                    color: black;
                    background: white;
                    padding: 4px;
                    border-radius: 5px;
                    border: 1px solid black;
                ">
                ⏱️ Total Time: {total_time} hrs
                </div>
                """
            )
        ).add_to(m)

    return m

# =============================
# UI
# =============================
cities = list(coords.keys())

start_dd = widgets.Dropdown(options=cities, description='Start:')
goal_dd = widgets.Dropdown(options=cities, description='Goal:')
button = widgets.Button(description="Find Route", button_style='success')
output = widgets.Output()

def on_click(b):
    with output:
        output.clear_output()

        start = start_dd.value
        goal = goal_dd.value

        print("🔎 SEARCHING BEST ROUTE...\n")

        bfs_path = bfs(start, goal)
        dfs_path = dfs(start, goal)
        ucs_path, cost, ttime = ucs(start, goal)

        print("BFS:", bfs_path)
        print("DFS:", dfs_path)

        print("\n🚀 UCS BEST PATH:", ucs_path)
        print("💰 Cost:", cost)
        print("⏱️ Estimated Time:", ttime, "hrs")

        directions(ucs_path)

        simulate_travel(ucs_path)

        print("\n🗺️ MAP VIEW:")

        m = show_map(ucs_path, ttime)
        display(HTML(m._repr_html_()))

button.on_click(on_click)

display(start_dd, goal_dd, button, output)

Dropdown(description='Start:', options=('Karachi', 'Hyderabad', 'Sukkur', 'Multan', 'Lahore', 'Islamabad', 'Pe…

Dropdown(description='Goal:', options=('Karachi', 'Hyderabad', 'Sukkur', 'Multan', 'Lahore', 'Islamabad', 'Pes…

Button(button_style='success', description='Find Route', style=ButtonStyle())

Output()